# 🤖 AI Agents under the Hood

**Ein Agent — *from scratch*.**

Kernbotschaft des Vortrags:

> Ein **LLM** ist nur *Text rein → Text raus*. Es hat kein Gedächtnis, keine Werkzeuge, keine Schleife.
> Ein **Agent** ist alles, was wir *drumherum* bauen: Memory, Tools, Loop, Context-Management und ein Harness, das das Ganze stabil hält.

| Kapitel | Frage | Zutat |
|---|---|---|
| 1 | Was *ist* ein LLM? | `llm()` |
| 2 | Warum „vergisst" es? | `messages[]` (Memory) |
| 3 | Wie bekommt es Werkzeuge? | Tool-Calling |
| 4 | Was ist *der* Agent? | Agentic Loop |
| 5 | Was heißt „Denken / Planen"? | ReAct & Planning |
| 6 | Echte Coding-Fähigkeiten | write / run / test (Sandbox) |
| 7 | Was steht im Kontextfenster? | Context Engineering |
| 8 | Was hält das stabil? | Harness Engineering |
| 9 | Alles zusammen | Coding-Agent Finale |

## 0 · Setup

Konfiguration kommt komplett aus der `.env` (Vorlage: `.env.example`). Abhängigkeiten einmalig mit **`uv sync`** installieren, dann `uv run jupyter lab` starten.

Die Funktion `llm()` unten ist unser **einziger Draht zum Modell** — sie begleitet uns durch alle 9 Kapitel.

In [1]:
import os, json, subprocess
from pathlib import Path
from openai import AzureOpenAI
from dotenv import load_dotenv

load_dotenv()  # liest die .env

client = AzureOpenAI(
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    api_version=os.environ.get("AZURE_OPENAI_API_VERSION", "2024-10-21"),
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
)
DEPLOYMENT = os.environ["AZURE_OPENAI_DEPLOYMENT"]

def llm(messages, tools=None, tool_choice="auto"):
    """Ein einziger Call zum Modell. Das ist ALLES, was ein LLM kann:
    Nachrichten rein -> eine Antwort raus. (Kein State, keine Tools von selbst.)"""
    kwargs = dict(model=DEPLOYMENT, messages=messages)
    if tools:
        kwargs["tools"] = tools
        kwargs["tool_choice"] = tool_choice
    return client.chat.completions.create(**kwargs)

print("Client bereit. Deployment:", DEPLOYMENT)

Client bereit. Deployment: gpt-5.4-mini


## 1 · Das nackte LLM — Text rein, Text raus

Ein LLM ist *stateless*. Es kennt nur die Nachrichten, die wir ihm **in genau diesem einen Call** mitgeben. Kein Zugriff auf Dateisystem, Netz oder Uhr.

Fragen wir nach etwas, das es nicht wissen kann, gibt es **zwei Ausgänge** — und beide sind ohne Tools wertlos:
- **bestenfalls ehrlich:** „Das kann ich nicht."
- **schlimmstenfalls erfunden:** es **rät** plausibel (Halluzination).

In [2]:
# Frage 1 — etwas, das Zugriff aufs Dateisystem bräuchte: hier ist es meist ehrlich.
print("— Kann es die Dateien sehen? —")
print(llm([{"role": "user", "content": "Liste die Dateien im aktuellen Verzeichnis auf."}]).choices[0].message.content)

# Frage 2 — etwas, das es nicht wissen KANN: hier rät es gern plausibel daneben.
print("\n— Was steht in der README? —")
print(llm([{"role": "user", "content": "Was steht wörtlich in der ersten Zeile der Datei README.md dieses Projekts?"}]).choices[0].message.content)

— Kann es die Dateien sehen? —
Ich kann das aktuelle Verzeichnis hier nicht direkt auslesen. Wenn du möchtest, kann ich dir aber den passenden Befehl nennen:

- **Linux/macOS:** `ls -la`
- **Windows (CMD):** `dir`
- **Windows (PowerShell):** `Get-ChildItem`

Wenn du mir die Ausgabe schickst, kann ich sie dir auch interpretieren.

— Was steht in der README? —
Ich habe keinen direkten Zugriff auf die Dateien dieses Projekts und kann daher die erste Zeile von `README.md` nicht wörtlich auslesen.

Wenn du mir den Inhalt von `README.md` hier einfügst oder die Datei hochlädst, sage ich dir sofort die erste Zeile.


👉 Das Modell *kann* die Dateien nicht sehen — es lehnt ehrlich ab oder erfindet eine plausible Antwort. **Kein Gedächtnis, keine Tools.** Beheben wir beides — Schritt für Schritt. **Zuerst das Gedächtnis.**

## 2 · Gedächtnis — die Konversation sind WIR

Das Modell merkt sich **nichts** zwischen zwei Calls. „Gedächtnis" ist keine Eigenschaft des Modells, sondern unsere Arbeit: *wir* hängen jede Nachricht an eine Liste `messages` an und schicken die **ganze Historie** bei jedem Call erneut mit.

In [3]:
# ❌ OHNE Memory: zwei getrennte Calls. Der 2. Call kennt den 1. nicht — jeder Call fängt bei null an.
print("Call 1:", llm([{"role": "user", "content": "Ich heiße Rudi. Merk dir das."}]).choices[0].message.content)
print("Call 2:", llm([{"role": "user", "content": "Wie heiße ich?"}]).choices[0].message.content)

Call 1: Verstanden, Rudi. Ich merke mir deinen Namen für diese Unterhaltung.
Call 2: Ich weiß leider nicht, wie du heißt. Wenn du möchtest, kannst du mir deinen Namen sagen — dann spreche ich dich gern so an.


In [5]:
# ✅ MIT Memory: WIR tragen die Historie selbst zusammen und schicken sie komplett mit.
messages = [
    {"role": "user", "content": "Ich heiße Rudi. Merk dir das."},
    {"role": "assistant", "content": "Klar, Rudi — ich merke mir deinen Namen."},
    {"role": "user", "content": "Wie heiße ich?"},
]
print(llm(messages).choices[0].message.content)

Du heißt Rudi.


## 3 · Tools — dem Modell Hände bzw. Werkzeuge geben

Memory ✓ — jetzt die **Tools**. Erinnerst du dich an Kapitel 1, wo das Modell die Dateien *nicht* sehen konnte? Das ändern wir jetzt.

Ein **Tool** ist eine ganz normale Python-Funktion. Dem Modell beschreiben wir sie als **JSON-Schema** (Name, Zweck, Parameter). Wichtig:

- Das Modell **führt nichts aus**. Es kann nur *sagen*: „Bitte ruf `list_files(path='.')` auf."
- Diese Aufforderung kommt als strukturierter **`tool_call`** zurück — nicht als Fließtext.
- **Wir** führen die Funktion aus und liefern das Ergebnis als neue Nachricht (`role: "tool"`) zurück.

In [6]:
def list_files(path="."):
    return "\n".join(sorted(os.listdir(path)))

list_files_schema = {
    "type": "function",
    "function": {
        "name": "list_files",
        "description": "Listet die Dateien in einem Verzeichnis auf.",
        "parameters": {
            "type": "object",
            "properties": {"path": {"type": "string", "description": "Pfad, Default '.'"}},
            "required": [],
        },
    },
}

resp = llm(
    [{"role": "user", "content": "Welche Dateien liegen im aktuellen Verzeichnis?"}],
    tools=[list_files_schema],
)
msg = resp.choices[0].message
print("content   :", msg.content)
print("tool_calls:", msg.tool_calls)

content   : None
tool_calls: [ChatCompletionMessageFunctionToolCall(id='call_0MHOG1Wqg0pARBLbtquXQ4Py', function=Function(arguments='{"path":"."}', name='list_files'), type='function')]


👉 `content` ist leer — stattdessen steht in `tool_calls`, **welche Funktion** das Modell mit **welchen Argumenten** aufgerufen haben will. Jetzt führen *wir* aus und geben das Ergebnis zurück. Achte auf die **zwei Runden** — genau dieses Hin und Her automatisieren wir im nächsten Kapitel zur Schleife:

In [7]:
messages = [{"role": "user", "content": "Welche Dateien liegen im aktuellen Verzeichnis?"}]

# Runde 1: Modell fordert das Tool an
resp = llm(messages, tools=[list_files_schema])
msg = resp.choices[0].message
tc = msg.tool_calls[0]
args = json.loads(tc.function.arguments or "{}")
result = list_files(**args)
print("Tool-Ergebnis:\n", result)

# Assistant-Nachricht (mit tool_call) + Tool-Ergebnis an die Historie hängen
messages.append({
    "role": "assistant", "content": msg.content,
    "tool_calls": [{"id": tc.id, "type": "function",
                    "function": {"name": tc.function.name, "arguments": tc.function.arguments}}],
})
messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})

# Runde 2: Modell formuliert die finale Antwort aus dem Tool-Ergebnis
final = llm(messages, tools=[list_files_schema])
print("\nFinale Antwort:\n", final.choices[0].message.content)

Tool-Ergebnis:
 .cursorignore
.env
.env.example
.gitignore
.ipynb_checkpoints
.pytest_cache
.venv
AI_Agents_under_the_Hood.ipynb
AI_Agents_under_the_Hood_Advanced.ipynb
README.md
agent_workspace
build_notebook.py
notebook_source.txt
pyproject.toml
setup.ps1
start-ctxman-azure-compaction.ps1
uv.lock
v1.json

Finale Antwort:
 Im aktuellen Verzeichnis liegen u. a. folgende Dateien und Ordner:

- `.cursorignore`
- `.env`
- `.env.example`
- `.gitignore`
- `.ipynb_checkpoints/`
- `.pytest_cache/`
- `.venv/`
- `AI_Agents_under_the_Hood.ipynb`
- `AI_Agents_under_the_Hood_Advanced.ipynb`
- `README.md`
- `agent_workspace/`
- `build_notebook.py`
- `notebook_source.txt`
- `pyproject.toml`
- `setup.ps1`
- `start-ctxman-azure-compaction.ps1`
- `uv.lock`
- `v1.json`

Wenn du willst, kann ich dir auch eine nach **Dateien** und **Verzeichnissen** getrennte Liste geben.


## 4 · Der Agentic Loop — *das* ist der Agent

Oben haben wir die zwei Runden von Hand gemacht. Ein Agent automatisiert genau das in einer **Schleife**:

```
solange das Modell ein Tool aufruft:
    Tool ausführen  ->  Ergebnis anhängen  ->  Modell erneut fragen
sonst:
    fertige Antwort zurückgeben
```

Mehr ist es im Kern **nicht**. Wir trennen das in zwei Teile: erst das **Drumherum** (eine Mini-Registry, damit Tools an einer Stelle leben), dann den **eigentlichen Agenten** — die Schleife, die exakt diesem Pseudocode entspricht.

In [9]:
# --- Drumherum (einmal aufsetzen): eine Mini-Registry für Tools ---
# Idee: jedes Tool = Python-Funktion + JSON-Schema, beides an EINER Stelle registriert.
TOOLS = []        # JSON-Schemas fürs Modell
DISPATCH = {}     # name -> Python-Funktion

def register(name, description, parameters):
    def deco(fn):
        TOOLS.append({"type": "function", "function": {
            "name": name, "description": description, "parameters": parameters}})
        DISPATCH[name] = fn
        return fn
    return deco

@register("list_files", "Listet Dateien in einem Verzeichnis.",
          {"type": "object", "properties": {"path": {"type": "string"}}, "required": []})
def _list_files(path="."):
    return "\n".join(sorted(os.listdir(path)))

#####################################################################################################


def to_assistant_dict(msg):
    """OpenAI-Antwortobjekt -> serialisierbares Dict für die Historie."""
    d = {"role": "assistant", "content": msg.content}
    if msg.tool_calls:
        d["tool_calls"] = [{"id": tc.id, "type": "function",
                            "function": {"name": tc.function.name, "arguments": tc.function.arguments}}
                           for tc in msg.tool_calls]
    return d

print("Werkzeugkasten bereit. Registrierte Tools:", list(DISPATCH))

Werkzeugkasten bereit. Registrierte Tools: ['list_files']


In [10]:
# ★★★  DAS IST DER AGENT  ★★★  — der komplette Loop, ~15 Zeilen
def run_agent(task, system=None, max_steps=10, verbose=True):
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": task})

    for step in range(1, max_steps + 1):
        msg = llm(messages, tools=TOOLS or None).choices[0].message
        messages.append(to_assistant_dict(msg))

        if not msg.tool_calls:                       # keine Tools mehr -> fertig
            if verbose: print(f"[Schritt {step}] ✓ finale Antwort")
            return msg.content, messages

        for tc in msg.tool_calls:                    # alle angeforderten Tools ausführen
            args = json.loads(tc.function.arguments or "{}")
            if verbose: print(f"[Schritt {step}] → {tc.function.name}({args})")
            try:
                result = DISPATCH[tc.function.name](**args)
            except Exception as e:
                result = f"ERROR: {e}"
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": str(result)})

    return "(max_steps erreicht)", messages


# Erste echte Agenten-Aufgabe: das Modell entscheidet selbst, list_files zu rufen.
# Liste alle Dateien im aktuellen Verzeichnis. Dann zähle die alle Files und zum schluss sag mir noch welche Dateien mit einem A beginnen
answer, _ = run_agent("Welche Dateien im aktuellen Verzeichnis beginnen mit dem Buchstaben A?")
print("\n", answer)

[Schritt 1] → list_files({'path': '.'})
[Schritt 2] ✓ finale Antwort

 Im aktuellen Verzeichnis beginnen diese Dateien mit **A**:

- `AI_Agents_under_the_Hood.ipynb`
- `AI_Agents_under_the_Hood_Advanced.ipynb`


## 5 · ReAct & Planning — was „Denken" wirklich heißt

Wir haben einen funktionierenden Agenten. Aber *wie* entscheidet er? **ReAct** = *Reasoning + Acting* verschränkt: Das Modell *begründet* kurz, *handelt* (Tool), *beobachtet* das Ergebnis, begründet erneut … bis das Ziel erreicht ist. „Denken" ist dabei nichts Magisches — es sind **Tokens, die das Modell vor der nächsten Aktion erzeugt** und die seine Folge-Entscheidung steuern.

**Wichtig für den roten Faden:** Der Loop bleibt *exakt* derselbe wie in Kapitel 4 — wir ändern nur den **System-Prompt** und machen die Zwischenschritte sichtbar. Zwei gängige Patterns:
- **ReAct** (reaktiv, schrittweise): denke → handle → beobachte → wiederhole.
- **Plan-and-Execute**: erst einen Plan erstellen, dann abarbeiten.

Unten machen wir **beides sichtbar**: bei ReAct die Gedanken-Spur (💭→🔧→👁), bei Plan-and-Execute den mitgeführten **Checklisten-Plan**.

In [11]:
REACT_SYSTEM = (
    "Arbeite nach dem ReAct-Muster (Reasoning + Acting). Schreibe VOR jedem Tool-Aufruf einen "
    "kurzen Gedanken (1 Satz) als normalen Text, rufe dann genau EIN Tool auf und nutze dessen "
    "Ergebnis für den nächsten Schritt. Wenn du genug weißt, antworte final."
)

@register("read_file", "Liest den Textinhalt einer Datei (erste 2000 Zeichen).",
          {"type": "object", "properties": {"path": {"type": "string"}}, "required": ["path"]})
def _read_file(path):
    return Path(path).read_text(encoding="utf-8", errors="replace")[:2000]

# Gleicher Loop wie run_agent — NEU ist nur: System-Prompt + sichtbare 💭/🔧/👁-Spur.
def run_react(task, max_steps=8):
    """ReAct sichtbar gemacht: 💭 Gedanke → 🔧 Aktion → 👁 Beobachtung, Schritt für Schritt."""
    messages = [{"role": "system", "content": REACT_SYSTEM}, {"role": "user", "content": task}]
    for step in range(1, max_steps + 1):
        msg = llm(messages, tools=TOOLS).choices[0].message
        messages.append(to_assistant_dict(msg))
        if not msg.tool_calls:                             # kein Tool mehr -> finale Antwort
            return msg.content
        if msg.content:                                    # der "Thought" VOR der Aktion
            print(f"[{step}] 💭 {msg.content.strip()}")
        for tc in msg.tool_calls:                          # "Action" + "Observation"
            args = json.loads(tc.function.arguments or "{}")
            print(f"     🔧 {tc.function.name}({args})")
            try:    
                result = str(DISPATCH[tc.function.name](**args))
            except Exception as e: result = f"ERROR: {e}"
            print(f"     👁 {result[:80].strip()} …")
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})
    return "(max_steps erreicht)"

print(run_react("Welche Python-Version verlangt dieses Projekt und welche Test-Bibliothek nutzt es?"))

[1] 💭 Ich prüfe die Projektdateien, um die Python-Version und die Test-Bibliothek zu finden.
     🔧 list_files({'path': '.'})
     👁 .cursorignore
.env
.env.example
.gitignore
.ipynb_checkpoints
.pytest_cache
.ven …
[2] 💭 Ich lese die Paketkonfiguration, weil dort die Python-Anforderung und Test-Abhängigkeiten stehen.
     🔧 read_file({'path': 'pyproject.toml'})
     👁 [project]
name = "ai-agents-under-the-hood"
version = "0.1.0"
description = "Ein …
Das Projekt verlangt **Python 3.10 oder neuer** (`requires-python = ">=3.10"`).

Als Test-Bibliothek nutzt es **pytest** (`pytest>=8.0.0`).


### Plan-and-Execute — den Plan *sehen* und abarbeiten

Bei **Plan-and-Execute** erstellt das Modell **zuerst einen expliziten Plan** und arbeitet ihn dann Schritt für Schritt ab. Damit das nicht im Verborgenen passiert, geben wir dem Agenten ein **`update_plan`-Tool**: Es führt eine **Markdown-Checkliste**, die das Modell selbst anlegt und nach jedem Schritt aktualisiert (`todo → doing → done`).

Genau so machen es echte Agents wie **Claude Code** (dort heißt das Tool `TodoWrite`). Der Plan wird zum sichtbaren, mitgeführten Artefakt — der Kernunterschied zu ReAct, das rein reaktiv Schritt für Schritt entscheidet.

In [14]:
from IPython.display import display, Markdown

PLAN = []   # der mitgeführte Plan: Liste von {"step", "status"}

def _show_plan():
    def line(s):
        if s["status"] == "done":  return f"- [x] {s['step']}"
        if s["status"] == "doing": return f"- [ ] 🔄 **{s['step']}**"
        return f"- [ ] {s['step']}"
    display(Markdown("**📋 Plan**\n\n" + "\n".join(line(s) for s in PLAN)))

@register("update_plan",
          "Erstellt oder aktualisiert den Arbeitsplan als Checkliste. Übergib IMMER die komplette "
          "Schrittliste mit Status je Schritt (todo, doing, done).",
          {"type": "object", "properties": {"steps": {"type": "array", "items": {
              "type": "object",
              "properties": {"step": {"type": "string"},
                             "status": {"type": "string", "enum": ["todo", "doing", "done"]}},
              "required": ["step", "status"]}}}, "required": ["steps"]})
def _update_plan(steps):
    global PLAN
    PLAN = steps
    _show_plan()                                           # Plan nach jeder Änderung neu rendern
    return "Plan aktualisiert: " + "; ".join(f"{s['step']}={s['status']}" for s in steps)

PLAN_SYSTEM = (
    "Du bist ein Plan-and-Execute-Agent. Rufe ZUERST update_plan mit allen nötigen Schritten auf "
    "(alle status='todo'). Arbeite die Schritte dann der Reihe nach ab: setze den aktuellen Schritt "
    "per update_plan auf 'doing', führe ihn mit den passenden Tools aus, markiere ihn dann als 'done' "
    "(erneut update_plan). Sind alle Schritte 'done', gib die finale Antwort."
)

# Wieder derselbe Loop — NEU ist nur das update_plan-Tool + ein anderer System-Prompt.
def run_plan(task, max_steps=12):
    """Wie run_agent, zeigt aber den Plan (statt der rohen update_plan-Argumente)."""
    messages = [{"role": "system", "content": PLAN_SYSTEM}, {"role": "user", "content": task}]
    for step in range(1, max_steps + 1):
        msg = llm(messages, tools=TOOLS).choices[0].message
        messages.append(to_assistant_dict(msg))
        if not msg.tool_calls:
            return msg.content
        for i, tc in enumerate(msg.tool_calls):
            #print(f"ToolCall #{i}")
            args = json.loads(tc.function.arguments or "{}")
            if tc.function.name != "update_plan":          # Tool-Aktionen zeigen; Plan rendert sich selbst
                print(f"🔧 {tc.function.name}({args})")
            try:    result = str(DISPATCH[tc.function.name](**args))
            except Exception as e: result = f"ERROR: {e}"
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})
    return "(max_steps erreicht)"

answer = run_plan("Lies pyproject.toml und .env.example und erkläre in zwei Sätzen, "
                  "was man zum Start dieses Projekts braucht.")
print("\n", answer)

**📋 Plan**

- [ ] Dateien pyproject.toml und .env.example finden und lesen
- [ ] Benötigte Startvoraussetzungen aus den Dateien ableiten
- [ ] Antwort in genau zwei Sätzen formulieren

🔧 read_file({'path': 'pyproject.toml'})
🔧 read_file({'path': '.env.example'})


**📋 Plan**

- [x] Dateien pyproject.toml und .env.example finden und lesen
- [ ] 🔄 **Benötigte Startvoraussetzungen aus den Dateien ableiten**
- [ ] Antwort in genau zwei Sätzen formulieren

**📋 Plan**

- [x] Dateien pyproject.toml und .env.example finden und lesen
- [x] Benötigte Startvoraussetzungen aus den Dateien ableiten
- [ ] 🔄 **Antwort in genau zwei Sätzen formulieren**

**📋 Plan**

- [x] Dateien pyproject.toml und .env.example finden und lesen
- [x] Benötigte Startvoraussetzungen aus den Dateien ableiten
- [x] Antwort in genau zwei Sätzen formulieren


 Zum Start brauchst du mindestens Python 3.10 sowie die in `pyproject.toml` genannten Abhängigkeiten, vor allem `openai`, `python-dotenv`, `tiktoken`, `jupyterlab` und `ipykernel`. Außerdem musst du eine `.env`-Datei aus `.env.example` anlegen und deine Azure-OpenAI-Zugangsdaten eintragen: Endpoint, API-Key, Deployment-Name und API-Version.


## 6 · Coding-Tools — Sandbox & Approval

Bisher durfte der Agent nur *lesen*. Jetzt geben wir ihm echte Coding-Fähigkeiten:
- **Dateien schreiben**
- **Shell/Code ausführen**
- **Tests laufen lassen**.

  
Macht braucht Leitplanken, also zwei Sicherheitsnetze:

1. **Sandbox**: Alles passiert im Unterordner `./agent_workspace`. Schreib-/Lese-Pfade werden darauf eingesperrt.
2. **Approval**: Vor jeder Shell-Ausführung fragt der Agent per `input()` um Erlaubnis (`j/N`).

Wir registrieren die neuen Tools einfach in *dieselbe* Registry — der Loop merkt davon nichts.

In [61]:
WORKSPACE = Path("./agent_workspace").resolve()
WORKSPACE.mkdir(exist_ok=True)
APPROVAL = True   # auf False setzen -> ohne Rückfrage (schnellere Demo)

def _safe(path):
    """Sperrt Pfade in die Sandbox ein."""
    p = (WORKSPACE / path).resolve()
    if not str(p).startswith(str(WORKSPACE)):
        raise ValueError(f"Pfad außerhalb der Sandbox: {path}")
    return p

# --- Tool: Datei schreiben (nur innerhalb der Sandbox) ---
@register("write_file", "Schreibt Text in eine Datei (innerhalb der Sandbox).",
          {"type": "object", "properties": {
              "path": {"type": "string"}, "content": {"type": "string"}},
           "required": ["path", "content"]})
def _write_file(path, content):
    p = _safe(path); p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(content, encoding="utf-8")
    return f"{len(content)} Zeichen nach {path} geschrieben."

# --- Tool: Datei aus der Sandbox lesen ---
@register("read_ws_file", "Liest eine Datei aus der Sandbox.",
          {"type": "object", "properties": {"path": {"type": "string"}}, "required": ["path"]})
def _read_ws_file(path):
    return _safe(path).read_text(encoding="utf-8", errors="replace")

# --- Tool: Shell-Befehl ausführen (mit Approval-Rückfrage) ---
@register("run_shell", "Führt einen PowerShell-Befehl in der Sandbox aus (stdout/stderr zurück).",
          {"type": "object", "properties": {"command": {"type": "string"}}, "required": ["command"]})
def _run_shell(command):
    if APPROVAL:
        ans = input(f"\n⚠️  Shell ausführen?\n  {command}\n[j/N] ")
        if ans.strip().lower() not in ("j", "ja", "y", "yes"):
            return "ABGELEHNT vom Benutzer."
    r = subprocess.run(["powershell", "-NoProfile", "-Command", command],
                       cwd=WORKSPACE, capture_output=True, text=True, timeout=120)
    out = f"exit={r.returncode}\n--- STDOUT ---\n{r.stdout}\n--- STDERR ---\n{r.stderr}"
    return out[:4000]

print("Sandbox:", WORKSPACE)
print("Registrierte Tools:", list(DISPATCH))

Sandbox: C:\Users\rudi\source\fsod\AI_Agents_Under_The_Hood\agent_workspace
Registrierte Tools: ['list_files', 'read_file', 'update_plan', 'write_file', 'read_ws_file', 'run_shell']


## 7 · Context Engineering — was ins Fenster passt

Je mehr der Agent tut, desto länger wird die Historie aus Kapitel 2 — und genau die schicken wir bei *jedem* Call komplett mit. Das Modell sieht pro Call nur das **Kontextfenster**: System-Prompt + komplette Message-Historie + Tool-Ergebnisse. Das ist endlich (Tokens kosten Geld & Zeit, und irgendwann ist das Fenster voll). **Context Engineering** = bewusst entscheiden, *was* drinsteht.

Drei Hebel, die wir zeigen:
- **Token-Budget messen** — wie voll ist das Fenster?
- **Tool-Ergebnisse kürzen** (truncation) — riesige Outputs nicht ungefiltert anhängen.
- **Compaction** — alte Historie zusammenfassen, statt sie komplett mitzuschleppen.

In [62]:
try:
    import tiktoken
    _enc = tiktoken.get_encoding("o200k_base")
    def count_tokens(messages):
        return sum(len(_enc.encode(str(m.get("content") or ""))) for m in messages)
except Exception:
    def count_tokens(messages):                      # Fallback: grobe Schätzung
        return sum(len(str(m.get("content") or "")) for m in messages) // 4

def truncate(text, limit=2000):
    return text if len(text) <= limit else text[:limit] + f"\n…[{len(text)-limit} Zeichen gekürzt]"

def compact(messages, keep_last=4):
    """Fasst alte Nachrichten zu einer kurzen Notiz zusammen; behält die letzten paar im Original."""
    if len(messages) <= keep_last + 1:
        return messages
    head, tail = messages[:-keep_last], messages[-keep_last:]
    summary = llm([{"role": "user",
        "content": "Fasse den folgenden Verlauf in 3 Stichpunkten zusammen:\n" +
                   json.dumps([{k: m.get(k) for k in ("role", "content")} for m in head], ensure_ascii=False)}]
    ).choices[0].message.content
    return [{"role": "system", "content": "Bisheriger Verlauf (komprimiert):\n" + summary}] + tail

# Demo: Tokens vor/nach Compaction
demo = [{"role": "user", "content": "Lang lang lang " * 200} for _ in range(6)]
print("vorher :", count_tokens(demo), "Tokens,", len(demo), "Nachrichten")
small = compact(demo)
print("nachher:", count_tokens(small), "Tokens,", len(small), "Nachrichten")

vorher : 3606 Tokens, 6 Nachrichten
nachher: 2477 Tokens, 5 Nachrichten


## 8 · Harness Engineering — was den Agenten stabil hält

Der Loop funktioniert — aber „funktioniert in der Demo" ist nicht „läuft zuverlässig". Das **Harness** ist alles um das Modell herum, das aus einem unzuverlässigen Text-Generator ein robustes Werkzeug macht — genau das, was z. B. Claude Code ausmacht:

- **`max_steps`** gegen Endlosschleifen
- **Fehler abfangen** und als Tool-Ergebnis zurückgeben (das Modell kann sich korrigieren)
- **Approval / Permissions** vor riskanten Aktionen (Kapitel 6)
- **Truncation & Compaction** (Kapitel 7) im Loop
- **Tracing / Logging**, damit man sieht, was passiert
- **Retries** bei transienten API-Fehlern

Wir gießen das in `run_agent_v2` — **derselbe Loop wie `run_agent`**, nur mit diesen Schutzschichten drumherum.

In [63]:
def run_agent_v2(task, system=None, max_steps=12, token_budget=6000, on_event=print):
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": task})

    for step in range(1, max_steps + 1):
        # Harness: Kontext klein halten
        if count_tokens(messages) > token_budget:
            messages = compact(messages)
            on_event(f"[harness] Kontext komprimiert -> {count_tokens(messages)} Tokens")

        # Harness: Retry bei transienten API-Fehlern
        for attempt in range(3):
            try:
                msg = llm(messages, tools=TOOLS or None).choices[0].message
                break
            except Exception as e:
                on_event(f"[harness] API-Fehler ({attempt+1}/3): {e}")
                if attempt == 2:
                    raise

        messages.append(to_assistant_dict(msg))
        if not msg.tool_calls:
            on_event(f"[schritt {step}] ✓ fertig")
            return msg.content, messages

        for tc in msg.tool_calls:
            args = json.loads(tc.function.arguments or "{}")
            on_event(f"[schritt {step}] → {tc.function.name}({truncate(json.dumps(args, ensure_ascii=False), 120)})")
            try:
                result = str(DISPATCH[tc.function.name](**args))
            except Exception as e:
                result = f"ERROR: {e}"          # Fehler ist auch ein Ergebnis -> Selbstkorrektur
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": truncate(result)})

    return "(max_steps erreicht)", messages

print("run_agent_v2 bereit.")

run_agent_v2 bereit.


## 9 · Finale — Der Coding-Agent

Jetzt schließt sich der Kreis: **derselbe Loop** aus Kapitel 4, plus Coding-Tools (6), Context-Management (7) und Harness (8). Mehr ist nicht dazugekommen — der Loop hat nur gelernt, mehr zu tun und das stabiler. Wir geben eine echte Aufgabe und lassen den Agenten **schreiben → ausführen → testen**.

> ⚠️ Beim Lauf erscheinen `input()`-Approval-Fragen (`j/N`) für jeden Shell-Befehl. Für eine schnellere Demo `APPROVAL = False` setzen.

In [65]:
CODING_SYSTEM = (
    "Du bist ein Coding-Agent und arbeitest ausschließlich in der Sandbox. "
    "Schreibe Code mit write_file, führe ihn mit run_shell (z. B. 'python ...') aus und teste mit pytest. "
    "Wenn ein Test fehlschlägt, lies die Fehlermeldung, korrigiere den Code und versuche es erneut. "
    "Erkläre am Ende kurz, was du gebaut hast."
)

task = (
    "Erstelle fizzbuzz.py mit einer Funktion fizzbuzz(n), die für die Zahlen 1..n die FizzBuzz-Regeln "
    "als Liste von Strings zurückgibt. Schreibe dazu test_fizzbuzz.py mit pytest-Tests "
    "(prüfe: 3 -> 'Fizz', 5 -> 'Buzz', 15 -> 'FizzBuzz', 1 -> '1'). "
    "Führe danach 'python -m pytest -q' aus, bis alle Tests grün sind."
)

answer, history = run_agent_v2(task, system=CODING_SYSTEM, max_steps=15)
print("\n=== Ergebnis ===\n", answer)

[schritt 1] → update_plan({"steps": [{"step": "Projektdateien prüfen", "status": "doing"}, {"step": "fizzbuzz.py implementieren", "status": "todo"
…[189 Zeichen gekürzt])


**📋 Plan**

- [ ] 🔄 **Projektdateien prüfen**
- [ ] fizzbuzz.py implementieren
- [ ] test_fizzbuzz.py schreiben
- [ ] pytest ausführen und Fehler beheben
- [ ] Abschluss kurz zusammenfassen

[schritt 2] → list_files({"path": "."})
[schritt 2] → read_file({"path": "fizzbuzz.py"})
[schritt 2] → read_file({"path": "test_fizzbuzz.py"})
[schritt 3] → write_file({"path": "fizzbuzz.py", "content": "def fizzbuzz(n):\n    result = []\n    for i in range(1, n + 1):\n        if i % 15 
…[245 Zeichen gekürzt])
[schritt 4] → write_file({"path": "test_fizzbuzz.py", "content": "from fizzbuzz import fizzbuzz\n\n\ndef test_fizzbuzz_3():\n    assert fizzbuzz(
…[227 Zeichen gekürzt])
[schritt 5] → update_plan({"steps": [{"step": "Projektdateien prüfen", "status": "done"}, {"step": "fizzbuzz.py implementieren", "status": "done"}
…[189 Zeichen gekürzt])


**📋 Plan**

- [x] Projektdateien prüfen
- [x] fizzbuzz.py implementieren
- [x] test_fizzbuzz.py schreiben
- [ ] 🔄 **pytest ausführen und Fehler beheben**
- [ ] Abschluss kurz zusammenfassen

[schritt 6] → run_shell({"command": "python -m pytest -q"})



⚠️  Shell ausführen?
  python -m pytest -q
[j/N]  j


[schritt 7] → update_plan({"steps": [{"step": "Projektdateien prüfen", "status": "done"}, {"step": "fizzbuzz.py implementieren", "status": "done"}
…[188 Zeichen gekürzt])


**📋 Plan**

- [x] Projektdateien prüfen
- [x] fizzbuzz.py implementieren
- [x] test_fizzbuzz.py schreiben
- [x] pytest ausführen und Fehler beheben
- [x] Abschluss kurz zusammenfassen

[schritt 8] ✓ fertig

=== Ergebnis ===
 Fertig — ich habe `fizzbuzz.py` mit `fizzbuzz(n)` erstellt und `test_fizzbuzz.py` mit pytest-Tests für 3, 5, 15 und 1 geschrieben.

`python -m pytest -q` läuft erfolgreich:
- 4 passed

Wenn du möchtest, kann ich als Nächstes noch eine kleine `README`-Erklärung oder zusätzliche Tests für Randfälle ergänzen.


### Mit git committen (commit live, push auskommentiert)
Der Agent kann auch versionieren. `commit` ist sicher; `push` braucht Remote + Auth und ist live riskant — daher bewusst auskommentiert.

In [ ]:
git_task = (
    "Initialisiere ein git-Repository in der Sandbox (falls noch nicht vorhanden), "
    "konfiguriere user.name='Demo Agent' und user.email='agent@example.com' lokal, "
    "füge alle Dateien hinzu und committe sie mit der Nachricht 'FizzBuzz vom Agent'. "
    "Zeige am Ende 'git log --oneline'."
    # "Pushe anschließend nach origin main."   # <- push bewusst auskommentiert (braucht Remote + Auth)
)
answer, _ = run_agent_v2(git_task, system=CODING_SYSTEM, max_steps=12)
print("\n", answer)

## Wrap-up — 3 Dinge zum Mitnehmen

1. **Ein Agent ist ein LLM in einer Schleife mit Tools.** Wir haben *eine* Schleife (`run_agent`, ~15 Zeilen) gebaut und sie durch 9 Kapitel wachsen lassen — sie hat sich nie grundlegend geändert. Den Rest macht das Modell.
2. **Die Kunst liegt im Drumherum:** *Context Engineering* (was sieht das Modell?) und *Harness Engineering* (was hält es stabil & sicher?) entscheiden über die Qualität.
3. **Tools sind nur Funktionen + Schema.** Einmal registriert, nutzt der Agent sie über denselben Loop — beliebig erweiterbar.

### Übungen fürs Publikum
- Ein neues Tool registrieren (z. B. `http_get`) und im Loop nutzen.
- `APPROVAL` / `max_steps` / `token_budget` variieren und das Verhalten beobachten.
- Plan-and-Execute vs. ReAct an derselben Aufgabe vergleichen.